In [11]:
import numpy as np
import pandas as pd
import synergy_dataset as sd

data_path = "./data/"
single_dataset = "van_de_Schoot_2018"

In [12]:
studies = pd.read_json("synergy_studies_validation.jsonl", lines=True)
studies_filtered = studies.sort_values("dataset_id").reset_index(drop=True)
report_order = studies_filtered["dataset_id"].unique()

recall_files = [
    "recalls_old1_nb.csv",
    "recalls_old1_svm.csv",
    "recalls_new2_nb.csv",
    "recalls_new2_svm.csv",
    "recalls_new2_mxbai_svm.csv",
    "recalls_new2_e5_svm.csv",
]
recall_types = [
    "ASR1.6 TF-IDF + NB",
    "ASR1.6 TF-IDF + SVM",
    "ASR2 TF-IDF + NB",
    "ASR2 TF-IDF + SVM",
    "ASR2 mxbai + SVM",
    "ASR2 E5 + SVM",
]

In [13]:
def get_total_relevant(dataset_id):
    if dataset_id in {"Moran_2021_corrected", "Muthu_2021_corrected"}:
        return pd.read_csv(f"../src/datasets/{dataset_id}_shuffled_raw.csv")[
            "label_included"
        ].sum()
    else:
        return sd.Dataset(dataset_id).to_frame()["label_included"].sum()


# Build the dictionary
total_relevant_dict = {
    dataset_id: get_total_relevant(dataset_id)
    for dataset_id in studies_filtered["dataset_id"].unique()
}

In [14]:
recall_dfs = [pd.read_csv(data_path + f) for f in recall_files]

# Add metadata to each DataFrame
for i, df in enumerate(recall_dfs):
    df["dataset_name"] = studies_filtered["dataset_id"].values
    df["Model"] = recall_types[i]
    df["prior_inclusions"] = studies_filtered["prior_inclusions"].apply(len)
    df["prior_exclusions"] = studies_filtered["prior_exclusions"].apply(len)
    df["simulation_id"] = df.groupby("dataset_name").cumcount() + 1

df_all = pd.concat(recall_dfs, ignore_index=True)

df_all_melted = df_all.melt(
    id_vars=[
        "dataset_name",
        "Model",
        "prior_inclusions",
        "prior_exclusions",
        "simulation_id",
    ],
    var_name="step",
    value_name="recall",
).dropna()

df_all_melted["step"] = df_all_melted["step"].astype(int)

df_all_melted["total_relevant"] = df_all_melted["dataset_name"].map(total_relevant_dict)

df_all_melted["relative_recall"] = df_all_melted["recall"] / (
    df_all_melted["total_relevant"] - df_all_melted["prior_inclusions"]
)

# Normalize step values to [0,1]
df_all_melted["relative_step"] = df_all_melted.groupby(["dataset_name", "Model"])[
    "step"
].transform(lambda x: x / x.max())

In [15]:
for recall_type in recall_types:
    df_ah = df_all_melted[df_all_melted["dataset_name"] == single_dataset]
    df_ah = df_ah[df_ah["Model"] == recall_type]

    df_ah["target_recall"] = df_ah["total_relevant"] - df_ah["prior_inclusions"]
    df_filtered = df_ah[df_ah["recall"] == df_ah["target_recall"]]
    lowest_step_df = (
        df_filtered.sort_values(["simulation_id", "step"])
        .groupby("simulation_id")
        .first()
        .reset_index()
    )

    res = lowest_step_df[
        ["simulation_id", "step", "recall", "total_relevant", "prior_inclusions"]
    ]
    print(recall_type)
    print(np.mean(res["step"].to_list()))
    print(np.std(res["step"].to_list()), "\n")

ASR1.6 TF-IDF + NB
542.1
189.47319071573162 

ASR1.6 TF-IDF + SVM
1167.8
498.82497932641667 

ASR2 TF-IDF + NB
596.6
204.18873622215307 

ASR2 TF-IDF + SVM
270.6
11.48216007552586 

ASR2 mxbai + SVM
266.8
25.837182508934674 

ASR2 E5 + SVM
269.9
2.879236009777594 

